In [ ]:
import os
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, random_split
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image

In [2]:
!git remote -v

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [ ]:
!git 

In [ ]:
class AugmentedECGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels  # DataFrame containing labels and corresponding IDs
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Extract the image path and ID
        image_path = self.image_paths[idx]
        image_id = os.path.basename(image_path).split('_')[1].split('.')[0]  # Extract ID from path (e.g., '000100')

        # Find the corresponding row in the labels DataFrame using the ID
        label_row = self.labels[self.labels['ID'] == int(image_id)]  # Assuming 'ID' column is integers

        if label_row.empty:
            raise ValueError(f"ID {image_id} not found in the labels DataFrame.")

        # Extract the label values (assuming they are in the other columns of the DataFrame)
        labels = label_row.iloc[0, 1:].values  # Skipping the ID column

        # Try to load the image
        try:
            image = Image.open(image_path).convert("L")
        except (IOError, OSError) as e:
            print(f"Error loading image {image_path}: {e}")
            return None  # Skip this image if it cannot be loaded

        # Convert image to numpy array
        image = np.array(image)

        # Apply transformations (ensure the transformation works with named arguments)
        if self.transform:
            augmented = self.transform(image=image)  # Use 'image=image' to pass it as a named argument
            image = augmented['image']

        return {"pixel_values": image, "labels": torch.tensor(labels, dtype=torch.float32)}

In [ ]:
# Custom Resize with Anti-Aliasing
class ResizeWithAntiAliasing(A.ImageOnlyTransform):
    def __init__(self, width, height, always_apply=False, p=1.0):
        super(ResizeWithAntiAliasing, self).__init__(always_apply, p)
        self.width = width
        self.height = height

    def apply(self, image, **params):
        # Convert the image to a PIL image, apply resizing with anti-aliasing
        image_pil = Image.fromarray(image)
        image_resized = image_pil.resize((self.width, self.height), Image.Resampling.LANCZOS)
        return np.array(image_resized)

In [ ]:
train_transform = A.Compose([
    ResizeWithAntiAliasing(width=224, height=224),  # Resize to 224x224
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),  # Apply shift, scale, rotate
    A.GaussianBlur(blur_limit=(3, 7), p=1.0),  # Apply Gaussian blur
    A.Normalize(mean=(0.5,), std=(0.5,)),  # Normalize for single channel
    ToTensorV2(),  # Convert to PyTorch tensor
])

In [ ]:
# Define transformations for validation (no augmentation)
val_transform = A.Compose([
    ResizeWithAntiAliasing(width=224, height=224),  # Resize to 224x224
    A.Normalize(mean=(0.5,), std=(0.5,)),  # Normalize for single channel
    ToTensorV2(),
])

In [ ]:
# Load the CSV file with labels
labels_df = pd.read_csv("/kaggle/input/bhf-data-science-centre-ecg-challenge/train_final.csv")

In [ ]:
with open('/kaggle/input/bhf-reference-files/valid_images_list.txt', 'r') as file:
    valid_image_paths = [line.strip() for line in file]

with open('/kaggle/input/bhf-reference-files/valid_test_images_list.txt', 'r') as file:
    valid_test_images = [line.strip() for line in file]

In [ ]:
# Create the datasets A training and validation
train_dataset = AugmentedECGDataset(valid_image_paths, labels_df, transform=train_transform)
val_dataset = AugmentedECGDataset(valid_test_images, labels_df, transform=val_transform)

In [ ]:
from torch.utils.data import DataLoader

batch_size = 10

# Create DataLoaders for training and validation datasets
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
print("Dataset read in!")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import torch.nn.functional as F

class ECGClassifier(nn.Module):
    def __init__(self, num_labels=5):
        super(ECGClassifier, self).__init__()
        
        # Use ResNet18 as backbone but modify first layer for grayscale
        self.model = models.resnet18(pretrained=True)
        
        # Modify first conv layer to accept grayscale
        first_conv_weights = self.model.conv1.weight.data
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.conv1.weight.data = torch.mean(first_conv_weights, dim=1, keepdim=True)
        
        # Remove the original classifier
        num_features = self.model.fc.in_features  # This will be 512 for ResNet18
        self.model.fc = nn.Identity()
        
        # ECG-specific feature extraction - maintain 512 channels to match attention
        self.ecg_features = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),  # Changed from 256 to 512
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        
        # Multi-head classifier - adjusted for 512 input features
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),  # Changed input from 256 to 512
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_labels)
        )
        
        # Attention mechanism - keeping same dimensions
        self.attention = nn.Sequential(
            nn.Linear(512, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )
        
    def forward(self, x):
        # Extract features using backbone
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)  # [batch_size, 512, H, W]
        
        # Apply attention
        b, c, h, w = x.size()
        features = x.view(b, c, -1)  # [batch_size, 512, H*W]
        attention_weights = self.attention(features.permute(0, 2, 1))  # [batch_size, H*W, 1]
        attention_weights = F.softmax(attention_weights, dim=1)
        attended_features = torch.bmm(features, attention_weights)  # [batch_size, 512, 1]
        attended_features = attended_features.squeeze(-1)  # [batch_size, 512]
        
        # ECG-specific feature extraction
        x = self.ecg_features(x)  # Now outputs [batch_size, 512, 1, 1]
        x = x.view(x.size(0), -1)  # [batch_size, 512]
        
        # Combine attended features with ECG features
        x = x + attended_features  # Now dimensions match: both are [batch_size, 512]
        
        # Classification
        x = self.classifier(x)
        return x

def get_optimizer(model, learning_rate=0.001):
    return torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.01,
        betas=(0.9, 0.999)
    )

def get_scheduler(optimizer, num_training_steps):
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.001,
        total_steps=num_training_steps,
        pct_start=0.3
    )

In [ ]:
print('Beginning model training')

In [ ]:
import torch
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
import logging

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
def train_model(model, train_dataloader, val_dataloader, 
                num_epochs=10, device="cuda", patience=5,
                save_path='best_model.pth'):
    
    # Initialize optimizer and scheduler
    optimizer = get_optimizer(model)
    num_training_steps = num_epochs * len(train_dataloader)
    scheduler = get_scheduler(optimizer, num_training_steps)

    criterion = nn.BCEWithLogitsLoss()

    # Initialize early stopping
    early_stopping = EarlyStopping(patience=patience)
    
    # Track best metrics
    best_val_accuracy = 0
    best_model_state = None
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'label_precision': [],
        'label_recall': [],
        'label_f1': []
    }
    
    print(f"Training on device: {device}")
    model = model.to(device)
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        total_train_loss = 0
        train_batches = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs} [Train]')
        
        for batch in train_batches:
            # Move batch to device
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            
            # Clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update weights
            optimizer.step()
            scheduler.step()
            
            # Update metrics
            total_train_loss += loss.item()
            train_batches.set_postfix({'loss': loss.item()})
        
        avg_train_loss = total_train_loss / len(train_dataloader)
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        all_preds = []
        all_labels = []
        
        val_batches = tqdm(val_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs} [Val]')
        
        with torch.no_grad():
            for batch in val_batches:
                inputs = batch["pixel_values"].to(device)
                labels = batch["labels"].to(device)
                
                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                # Get predictions
                preds = (torch.sigmoid(outputs) > 0.5).float()
                
                # Update metrics
                total_val_loss += loss.item()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # Calculate validation metrics
        avg_val_loss = total_val_loss / len(val_dataloader)
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        
        val_accuracy = accuracy_score(all_labels.flatten(), all_preds.flatten())
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels.flatten(), 
            all_preds.flatten(), 
            average='binary'
        )
        
        # Compute label-specific metrics
        label_precision, label_recall, label_f1, _ = precision_recall_fscore_support(
            all_labels, 
            all_preds, 
            average=None
        )
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['val_precision'].append(precision)
        history['val_recall'].append(recall)
        history['val_f1'].append(f1)
        history['label_precision'].append(label_precision.tolist())
        history['label_recall'].append(label_recall.tolist())
        history['label_f1'].append(label_f1.tolist())
        
        # Save best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
            torch.save(best_model_state, save_path)
        
        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f"\nEpoch {epoch + 1}/{num_epochs} Summary:")
        print(f"Time: {epoch_time:.2f}s")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"Val Accuracy: {val_accuracy:.4f}")
        print(f"Val Precision: {precision:.4f}")
        print(f"Val Recall: {recall:.4f}")
        print(f"Val F1: {f1:.4f}")
        print(f"Label-wise Precision: {label_precision}")
        print(f"Label-wise Recall: {label_recall}")
        print(f"Label-wise F1: {label_f1}")
        
        # Early stopping check
        early_stopping(avg_val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break
    
    # Load best model
    model.load_state_dict(torch.load(save_path))
    return model, history


In [ ]:

# Run the training
if __name__ == "__main__":
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Initialize model and move to device
    model = ECGClassifier(num_labels=5)
    
    # Train model
    trained_model, history = train_model(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        num_epochs=10,
        device=device,
        patience=5,
        save_path='best_ecg_model.pth'
    )
    
    # Save training history
    np.save('training_history.npy', history)